In [2]:
# Install dependencies
!pip install statsbombpy pyarrow boto3 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.2 MB/s eta 0:00:00


In [3]:
import os
import io
import warnings
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import boto3
from concurrent.futures import ThreadPoolExecutor, as_completed
from statsbombpy import sb
from statsbombpy.api_client import NoAuthWarning
from google.colab import userdata

warnings.filterwarnings("ignore", category=NoAuthWarning)

In [4]:
# Load R2 credentials from Colab secrets
# Add these in the key icon (left sidebar) before running:
#   R2_ACCESS_KEY_ID, R2_SECRET_ACCESS_KEY, R2_ACCOUNT_ID, R2_BUCKET
account_id = userdata.get("R2_ACCOUNT_ID")
access_key = userdata.get("R2_ACCESS_KEY_ID")
secret_key = userdata.get("R2_SECRET_ACCESS_KEY")
bucket     = userdata.get("R2_BUCKET")

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{account_id}.r2.cloudflarestorage.com",
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name="auto"
)

# Verify connection
resp = s3.list_objects_v2(Bucket=bucket, MaxKeys=1)
print(f"Connected to R2 bucket: {bucket}")

Connected to R2 bucket: footydata


In [ ]:
def upload_parquet(df, key):
    """Serialize a DataFrame to Parquet in memory and upload to R2."""
    buf = io.BytesIO()
    pq.write_table(pa.Table.from_pandas(df, preserve_index=False), buf)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key=key, Body=buf)


def fetch_events_for_match(match_id):
    """Fetch and clean events for a single match. Returns (match_id, df) or (match_id, None) on failure."""
    try:
        df = sb.events(match_id=match_id)
        df["match_id"] = match_id
        if "location" in df.columns:
            df["location_x"] = df["location"].apply(lambda l: l[0] if isinstance(l, list) else None)
            df["location_y"] = df["location"].apply(lambda l: l[1] if isinstance(l, list) else None)
            df = df.drop(columns=["location"])
        df = df.rename(columns={
            "type":         "type_name",
            "play_pattern": "play_pattern_name",
            "team":         "team_name",
            "player":       "player_name",
            "position":     "position_name"
        })
        return match_id, df
    except Exception as e:
        print(f"  FAILED match {match_id}: {e}")
        return match_id, None

In [ ]:
# Fetch all male competitions and their matches
comps = sb.competitions()
comps = comps[comps["competition_gender"] == "male"].reset_index(drop=True)
print(f"{len(comps)} competition-seasons to ingest")

all_matches = []
for _, row in comps.iterrows():
    try:
        matches = sb.matches(competition_id=row["competition_id"], season_id=row["season_id"])
        matches["competition_id"] = row["competition_id"]
        all_matches.append(matches)
    except Exception as e:
        print(f"  FAILED competition {row['competition_id']} season {row['season_id']}: {e}")

matches_df = pd.concat(all_matches, ignore_index=True)
print(f"{len(matches_df)} total matches")

67 competition-seasons to ingest
2651 total matches


In [ ]:
for col in matches_df.select_dtypes(include='object').columns:
  matches_df[col] = matches_df[col].where(matches_df[col].isna(), matches_df[col].astype(str))

In [ ]:
# Upload competitions and matches to R2
upload_parquet(comps, "competitions/competitions.parquet")
print("competitions uploaded")

for comp_id, group in matches_df.groupby("competition_id"):
    upload_parquet(group, f"matches/competition_id={comp_id}/data_0.parquet")
print(f"matches uploaded ({matches_df['competition_id'].nunique()} partitions)")

competitions uploaded
matches uploaded (17 partitions)


In [ ]:
  # Find which competition_id folders already exist in R2
paginator = s3.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=bucket, Prefix='events/', Delimiter='/')

completed_comp_ids = set()
for page in pages:
    for prefix in page.get('CommonPrefixes', []):
        # prefix looks like 'events/competition_id=9/'
        part = prefix['Prefix'].split('competition_id=')[-1].strip('/')
        completed_comp_ids.add(int(part))
print(f"{len(completed_comp_ids)} competitions already on R2: {sorted(completed_comp_ids)}")

9 competitions already on R2: [9, 11, 16, 43, 87, 223, 1238, 1267, 1470]


In [ ]:
print(comps[["competition_id", "competition_name", "season_name"]].to_string())

    competition_id        competition_name season_name
0                9           1. Bundesliga   2023/2024
1                9           1. Bundesliga   2015/2016
2             1267  African Cup of Nations        2023
3               16        Champions League   2018/2019
4               16        Champions League   2017/2018
5               16        Champions League   2016/2017
6               16        Champions League   2015/2016
7               16        Champions League   2014/2015
8               16        Champions League   2013/2014
9               16        Champions League   2012/2013
10              16        Champions League   2011/2012
11              16        Champions League   2010/2011
12              16        Champions League   2009/2010
13              16        Champions League   2008/2009
14              16        Champions League   2006/2007
15              16        Champions League   2004/2005
16              16        Champions League   2003/2004
17        

In [ ]:
# Ingest events — one competition at a time to keep memory low
# ThreadPoolExecutor fetches matches within each competition in parallel
failed = []

for _, comp_row in comps.iterrows():
    comp_id = comp_row["competition_id"]
    season_id = comp_row["season_id"]
    comp_name = comp_row["competition_name"]

    if comp_id in completed_comp_ids:
        print(f"Skipping {comp_name} (season {season_id}) — already on R2")
        continue

    match_ids = matches_df[
        (matches_df["competition_id"] == comp_id) &
        (matches_df["season_id"] == season_id)
    ]["match_id"].tolist()

    if not match_ids:
        continue

    print(f"Fetching {len(match_ids)} matches — {comp_name} (season {season_id})")

    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(fetch_events_for_match, mid): mid for mid in match_ids}
        for future in as_completed(futures):
            match_id, df = future.result()
            if df is None:
                failed.append(match_id)
                continue
            df["competition_id"] = comp_id
            upload_parquet(df, f"events/competition_id={comp_id}/match_{match_id}.parquet")

    print(f"  done")

print(f"\nDone. {len(failed)} failed matches: {failed}")

Skipping 1. Bundesliga (season 281) — already on R2
Skipping 1. Bundesliga (season 27) — already on R2
Skipping African Cup of Nations (season 107) — already on R2
Skipping Champions League (season 4) — already on R2
Skipping Champions League (season 1) — already on R2
Skipping Champions League (season 2) — already on R2
Skipping Champions League (season 27) — already on R2
Skipping Champions League (season 26) — already on R2
Skipping Champions League (season 25) — already on R2
Skipping Champions League (season 24) — already on R2
Skipping Champions League (season 23) — already on R2
Skipping Champions League (season 22) — already on R2
Skipping Champions League (season 21) — already on R2
Skipping Champions League (season 41) — already on R2
Skipping Champions League (season 39) — already on R2
Skipping Champions League (season 37) — already on R2
Skipping Champions League (season 44) — already on R2
Skipping Champions League (season 76) — already on R2
Skipping Champions League (se

In [ ]:
# Smoke test — count rows in R2 for a quick sanity check
events_count  = s3.list_objects_v2(Bucket=bucket, Prefix="events/")["KeyCount"]
matches_count = s3.list_objects_v2(Bucket=bucket, Prefix="matches/")["KeyCount"]
print(f"events partitions on R2:  {events_count}")
print(f"matches partitions on R2: {matches_count}")

events partitions on R2:  1000
matches partitions on R2: 17


In [ ]:
paginator = s3.get_paginator('list_objects_v2')

events_count = sum(
    page['KeyCount']
    for page in paginator.paginate(Bucket=bucket, Prefix='events/')
)
matches_count = sum(
    page['KeyCount']
    for page in paginator.paginate(Bucket=bucket, Prefix='matches/')
)

print(f"events files:  {events_count}")
print(f"matches files: {matches_count}")

events files:  1357
matches files: 17
